# Qwen3-8B Quantization Research Benchmark

Reusable benchmark notebook for the quantization research project.

**Main rule:** keep the benchmark code fixed and change only `METHOD` in the configuration cell.

Current target:
- Qwen3-8B
- RTX 5050 Laptop GPU (~8 GB VRAM)
- Windows + CUDA-enabled PyTorch

Planned experiment sequence:
1. BF16 baseline
2. INT8 weight-only baseline
3. INT4 weight-only baseline
4. GPTQ INT4
5. AWQ INT4

GPTQ/AWQ calibration and checkpoint creation are separate stages; once those checkpoints exist, this same benchmark can load them without changing the measurement code.


In [1]:
# =========================
# CONFIGURATION
# =========================

MODEL_PATH = r"..\models\Qwen3-8B"

# CHANGE ONLY THIS VALUE
METHOD = "INT4_WO"
# BF16
# INT8_WO
# INT4_WO
# GPTQ_INT4
# AWQ_INT4

GROUP_SIZE = 128

PROMPT = "Explain what LLM quantization is in simple terms."
MAX_NEW_TOKENS = 128
WARMUP_RUNS = 1
BENCHMARK_RUNS = 3

print("METHOD:", METHOD)
print("MODEL:", MODEL_PATH)


METHOD: INT4_WO
MODEL: ..\models\Qwen3-8B


## 0. Environment check

In [2]:
import gc
import sys
import time
import json
import platform
import statistics
from pathlib import Path

import torch
import transformers

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(p.total_memory / 1024**3, 2), "GB")

print("OS:", platform.platform())
print("Model path exists:", Path(MODEL_PATH).exists())


c:\Users\shyle\OneDrive\Documents\Projects\Research_Works_LLM\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.9
PyTorch: 2.11.0+cu128
Transformers: 5.17.0
CUDA available: True
Torch CUDA: 12.8
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
VRAM: 7.96 GB
OS: Windows-10-10.0.26200-SP0
Model path exists: True


In [3]:
def bytes_to_gb(x):
    return x / (1024**3)

def directory_size_bytes(path):
    total = 0
    for p in Path(path).rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except OSError:
                pass
    return total

def reset_cuda_stats():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def peak_vram_gb():
    return bytes_to_gb(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else 0.0

def print_device_summary(model):
    dm = getattr(model, "hf_device_map", None)
    if dm:
        from collections import Counter
        for device, count in Counter(str(v) for v in dm.values()).items():
            print(f"{device}: {count} module(s)")
    else:
        print("Device map: not provided")


## 1. Model loader

The `METHOD` selector controls the loading/quantization path.

- `BF16`: original local checkpoint.
- `INT8_WO`: torchao INT8 weight-only baseline.
- `INT4_WO`: torchao INT4 group-wise weight-only baseline.
- `GPTQ_INT4` / `AWQ_INT4`: load local quantized checkpoints once created.

This keeps the benchmark protocol identical across methods. Current PyTorch torchao provides INT8 and INT4 weight-only quantization configurations; Hugging Face documents GPTQ and AWQ as separate PTQ methods. citeturn675140search0turn675140search5turn877807search0


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model(method):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    if method == "BF16":
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            dtype=torch.bfloat16,
            device_map="auto",
            low_cpu_mem_usage=True,
        )
        return tokenizer, model

    if method in {"INT8_WO", "INT4_WO"}:
        try:
            from torchao.quantization import (
                quantize_,
                Int8WeightOnlyConfig,
                Int4WeightOnlyConfig,
            )
        except ImportError as exc:
            raise ImportError(
                "Install torchao first: pip install torchao"
            ) from exc

        # Load the original checkpoint on CPU, then quantize.
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            dtype=torch.bfloat16,
            device_map={"": "cpu"},
            low_cpu_mem_usage=True,
        )

        if method == "INT8_WO":
            qconfig = Int8WeightOnlyConfig()
        else:
            qconfig = Int4WeightOnlyConfig(group_size=GROUP_SIZE)

        quantize_(model, qconfig)

        # Move quantized model to GPU for the benchmark.
        model = model.to("cuda")
        return tokenizer, model

    checkpoint = {
        "GPTQ_INT4": Path(r".\models\Qwen3-8B-GPTQ-INT4"),
        "AWQ_INT4": Path(r".\models\Qwen3-8B-AWQ-INT4"),
    }.get(method)

    if checkpoint is None:
        raise ValueError(f"Unsupported METHOD: {method}")

    if not checkpoint.exists():
        raise FileNotFoundError(
            f"{method} checkpoint not found at {checkpoint}. "
            "Create that quantized checkpoint first."
        )

    model = AutoModelForCausalLM.from_pretrained(
        checkpoint,
        dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    return tokenizer, model

reset_cuda_stats()
start_load = time.perf_counter()
tokenizer, model = load_model(METHOD)
load_seconds = time.perf_counter() - start_load

print(f"Load time: {load_seconds:.2f} s")
print_device_summary(model)


W0921 08:21:09.531000 48688 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 399/399 [00:27<00:00, 14.48it/s] 


ImportError: Requires mslk >= 1.0.0

## 2. Standardized generation benchmark

Do not change this cell between experiments. The same prompt, token budget, warmup and number of runs are used for every method.

In [ ]:
print("===== MODEL DEVICE / DTYPE INSPECTION =====")

print("First parameter:")
for name, param in model.named_parameters():
    print("Name :", name)
    print("Shape:", tuple(param.shape))
    print("Dtype:", param.dtype)
    print("Device:", param.device)
    print("Bytes:", param.numel() * param.element_size())
    break

print("\nCUDA memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")

print("CUDA memory reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")

print("CUDA max allocated:",
      round(torch.cuda.max_memory_allocated() / 1024**3, 3), "GB")

print("\nParameter dtype counts:")

from collections import Counter

dtype_counts = Counter(
    str(param.dtype)
    for param in model.parameters()
)

for dtype, count in dtype_counts.items():
    print(dtype, ":", count)

In [ ]:
print("\n===== MODEL STORAGE ESTIMATE =====")

total_parameter_bytes = 0

for name, param in model.named_parameters():
    total_parameter_bytes += param.numel() * param.element_size()

print(
    "Parameter storage estimate:",
    round(total_parameter_bytes / 1024**3, 3),
    "GB"
)

In [ ]:
# ============================================================
# VERIFY ACTUAL TORCHAO QUANTIZATION
# ============================================================

from collections import Counter

print("===== TORCHAO QUANTIZATION INSPECTION =====")

# 1. torchao version
try:
    import torchao
    print("torchao version:", torchao.__version__)
except Exception:
    print("torchao version: unable to read")

# 2. Inspect weight/module types
module_weight_types = Counter()
quantized_modules = []

for name, module in model.named_modules():
    if hasattr(module, "weight") and module.weight is not None:

        weight = module.weight

        module_type = type(module).__name__
        weight_type = type(weight).__name__
        weight_module = type(weight).__module__

        key = f"{module_type} -> {weight_module}.{weight_type}"
        module_weight_types[key] += 1

        # Look for torchao / quantization-specific tensor types
        text = f"{weight_module}.{weight_type}".lower()

        if "torchao" in text or "quant" in text:
            quantized_modules.append(
                (name, module_type, weight_module, weight_type, str(weight.dtype))
            )

print("\nWeight type summary:")
for key, count in module_weight_types.items():
    print(f"{count:4d}  {key}")

print("\nPotentially quantized weights:", len(quantized_modules))

print("\nFirst 15 potentially quantized modules:")
for item in quantized_modules[:15]:
    print(item)

# 3. Inspect a few Linear layers directly
print("\n===== SAMPLE LINEAR LAYERS =====")

count = 0

for name, module in model.named_modules():

    if isinstance(module, torch.nn.Linear):

        weight = module.weight

        print("\nLayer:", name)
        print("Module type :", type(module))
        print("Weight type :", type(weight))
        print("Weight module:", type(weight).__module__)
        print("Weight dtype:", weight.dtype)
        print("Weight device:", weight.device)
        print("Weight shape:", tuple(weight.shape))

        count += 1

        if count >= 5:
            break

In [ ]:
def prepare_inputs(text):
    inputs = tokenizer(text, return_tensors="pt")
    device = model.get_input_embeddings().weight.device
    return {k: v.to(device) for k, v in inputs.items()}

def generate_once(text):
    inputs = prepare_inputs(text)
    input_tokens = inputs["input_ids"].shape[-1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    output_tokens = output.shape[-1] - input_tokens
    tok_per_sec = output_tokens / elapsed if output_tokens else 0.0

    text_out = tokenizer.decode(
        output[0][input_tokens:],
        skip_special_tokens=True,
    )

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "time_s": elapsed,
        "decode_tok_s": tok_per_sec,
        "text": text_out,
    }

# Warm-up
for _ in range(WARMUP_RUNS):
    generate_once(PROMPT)

# Timed runs
reset_cuda_stats()
runs = [generate_once(PROMPT) for _ in range(BENCHMARK_RUNS)]

times = [r["time_s"] for r in runs]
rates = [r["decode_tok_s"] for r in runs]

print("===== BENCHMARK =====")
print("Method:", METHOD)
print("Model directory size:", f"{bytes_to_gb(directory_size_bytes(MODEL_PATH)):.2f} GB")
print("Mean generation time:", f"{statistics.mean(times):.3f} s")
print("Median generation time:", f"{statistics.median(times):.3f} s")
print("Mean decode speed:", f"{statistics.mean(rates):.2f} tok/s")
print("Peak VRAM:", f"{peak_vram_gb():.2f} GB")
print("Output tokens:", runs[-1]["output_tokens"])
print("\nResponse:")
print(runs[-1]["text"])


## 3. Save raw result

Each run creates a JSON record so the numbers can be transferred to the research workbook later.

In [ ]:
RESULTS_DIR = Path(r"..\results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

record = {
    "experiment_id": time.strftime("EXP-%Y%m%d-%H%M%S"),
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model": "Qwen3-8B",
    "method": METHOD,
    "bits": None if METHOD == "BF16" else (8 if METHOD == "INT8_WO" else 4),
    "group_size": None if METHOD in {"BF16", "INT8_WO"} else GROUP_SIZE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "vram_gb": bytes_to_gb(torch.cuda.get_device_properties(0).total_memory)
        if torch.cuda.is_available() else None,
    "software": {
        "python": sys.version.split()[0],
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "torch_cuda": torch.version.cuda,
    },
    "metrics": {
        "model_directory_size_gb": bytes_to_gb(directory_size_bytes(MODEL_PATH)),
        "load_time_s": load_seconds,
        "peak_vram_gb": peak_vram_gb(),
        "mean_generation_time_s": statistics.mean(times),
        "median_generation_time_s": statistics.median(times),
        "mean_decode_tok_s": statistics.mean(rates),
        "output_tokens": runs[-1]["output_tokens"],
    },
    "protocol": {
        "prompt": PROMPT,
        "max_new_tokens": MAX_NEW_TOKENS,
        "warmup_runs": WARMUP_RUNS,
        "benchmark_runs": BENCHMARK_RUNS,
    },
}

path = RESULTS_DIR / f"{record['experiment_id']}_{METHOD}.json"
path.write_text(json.dumps(record, indent=2), encoding="utf-8")
print("Saved:", path)


## 4. Research checklist

Before comparing methods, confirm:

- Same Qwen3-8B checkpoint
- Same prompt/evaluation set
- Same output-token limit
- Same warmup and run count
- Same hardware/software environment
- No unintended disk offload for a GPU-resident claim
- Model size measured from the actual artifact
- Quality metrics recorded separately
- Backend/kernel limitations documented

**Important:** lower precision does not automatically mean faster inference. Speed depends on the hardware and inference kernels, so memory and throughput must be measured separately. 
